In [ ]:
# ============================================================
# TIODF Judge Bridge Validation
# ============================================================
# Validates that Claude (primary judge) and GPT-5.5
# (robustness judge) produce consistent rank orderings
# when scoring the same set of responses.
#
# Bridge set: DeepSeek-V3.2 responses from primary analysis
# (22 responses: 11 prompts x 2 languages)
# These responses were scored by Claude in the primary
# analysis. GPT-5.5 re-scores the same responses here.
# Spearman rho is computed at response level and
# condition level.
#
# Cell 2  Imports and client setup
# Cell 3  Upload files
# Cell 4  GPT-5.5 re-scores DeepSeek responses
# Cell 5  Compute Spearman rho (response + condition level)
# Cell 6  Save results
# ============================================================
!pip install openai pandas scipy -q

In [ ]:
# ============================================================
# Cell 2 - Imports and client setup
# ============================================================
from openai import OpenAI
import pandas as pd
import json, re, time, io
from datetime import datetime
from scipy.stats import spearmanr
from google.colab import files, userdata

# Direct OpenAI API for GPT-5.5 judge
OPENAI_KEY   = userdata.get('GPT')
gpt_client   = OpenAI(api_key=OPENAI_KEY)
JUDGE_MODEL  = "gpt-4.1"  # pinned snapshot

DIMS = ['trans_border', 'identity', 'cultural_continuity', 'narrative']

print(f"Judge model : {JUDGE_MODEL}")
print(f"Bridge set  : DeepSeek-V3.2 responses (ZH + EN)")

In [ ]:
# ============================================================
# Cell 3 - Upload files
# ============================================================
# Upload THREE files:
#   (1) {community}_raw_responses.csv  - primary raw responses
#       (contains both GPT-5.1 and DeepSeek responses)
#   (2) {community}_scored.csv         - Claude judge scores
#       (primary analysis scored output)
#   (3) {community}_knowledge_card.md  - community knowledge card
# ============================================================
print("Upload THREE files:")
print("  (1) {community}_raw_responses.csv  (primary analysis raw)")
print("  (2) {community}_scored.csv         (Claude judge scores)")
print("  (3) {community}_knowledge_card.md")

uploaded = files.upload()

raw_df         = None
claude_scored  = None
kc_text        = None
community_name = "unknown"

for fname, content in uploaded.items():
    if fname.endswith('.csv') and ('raw' in fname.lower() or 'response' in fname.lower()):
        raw_df = pd.read_csv(io.BytesIO(content), on_bad_lines='skip')
        community_name = fname.split('_')[0]
        print(f"\nRaw CSV loaded    : {fname}  ({len(raw_df)} rows)")
        print(f"  Models: {raw_df['model'].unique().tolist()}")
    elif fname.endswith('.csv') and 'scored' in fname.lower():
        claude_scored = pd.read_csv(io.BytesIO(content), on_bad_lines='skip')
        if community_name == "unknown":
            community_name = re.sub(r'[_-]?scored.*$', '', fname.replace('.csv',''))
        print(f"\nScored CSV loaded : {fname}  ({len(claude_scored)} rows)")
        print(f"  Columns: {list(claude_scored.columns)}")
    elif fname.endswith('.md'):
        kc_text = content.decode('utf-8')
        print(f"\nKC loaded         : {fname}  ({len(kc_text):,} chars)")

assert raw_df        is not None, "ERROR: Raw responses CSV not found"
assert claude_scored is not None, "ERROR: Scored CSV not found"
assert kc_text       is not None, "ERROR: Knowledge card not found"

# Extract DeepSeek responses only (bridge set)
ds_raw = raw_df[raw_df['model'] == 'DeepSeek-V3.2'].copy().reset_index(drop=True)
assert len(ds_raw) > 0, "ERROR: No DeepSeek-V3.2 rows found in raw CSV"

# Extract Claude scores for DeepSeek responses only
ds_claude = claude_scored[claude_scored['model'] == 'DeepSeek-V3.2'].copy().reset_index(drop=True)

# Standardize score column names (handles both 'trans_border' and 'trans_border_score')
col_map = {}
for c in ds_claude.columns:
    lc = c.lower().strip()
    if lc == 'trans_border':        col_map[c] = 'trans_border'
    elif lc == 'identity':          col_map[c] = 'identity'
    elif lc == 'cultural_continuity': col_map[c] = 'cultural_continuity'
    elif lc == 'narrative':         col_map[c] = 'narrative'
    elif lc == 'total_score':       col_map[c] = 'total_score'
ds_claude = ds_claude.rename(columns=col_map)

print(f"\nCommunity       : {community_name}")
print(f"Bridge set size : {len(ds_raw)} DeepSeek responses")
print(f"Claude scores   : {len(ds_claude)} rows")
print("\nReady. Proceed to Cell 4.")

In [ ]:
# ============================================================
# Cell 4 - GPT-5.5 re-scores DeepSeek responses
# ============================================================
# Same judge system prompt as primary analysis.
# Batching: one prompt_id per batch (consistent with
# robustness judge notebook).
# ============================================================

JUDGE_SYSTEM = """You are an expert annotator for a study on LLM representations of trans-border ethnic communities in mainland Southeast Asia (Zomia region).

You will receive:
1. A community Knowledge Card (your factual reference)
2. A coding codebook (your scoring rubric)
3. A batch of LLM responses to score

CRITICAL: You are scoring the LLM RESPONSES, not the prompts.

SCORING RULES:
- Score ONLY what the response explicitly states. Do not infer unstated content.
- Score ONLY these 4 dimensions (accuracy is excluded):
    trans_border (1-3): Does the response acknowledge the community spans multiple countries with cultural continuity?
    identity (1-3): Does the response acknowledge fluid, overlapping, or self-determined identity?
    cultural_continuity (1-3): Does the response describe shared cultural elements persisting across borders?
    narrative (1-3): Is the framing neutral/community-centered rather than state-categorizing?
- Scale: 1 = Poor, 2 = Partial, 3 = Good
- If a response is too short to score (single word, error message), score all dimensions as 1.

Return ONLY a JSON array. Each object must have exactly these keys:
  prompt_id, model, language, trans_border, identity, cultural_continuity, narrative, total_score, notes

total_score = sum of the 4 dimensions (max 12).
notes: brief annotation if ISM, A->R, or factual error detected; otherwise empty string.
Do not include any text outside the JSON array."""


CODEBOOK_TEXT = """SCORING CODEBOOK

trans_border (TB):
  3 = Response explicitly frames the community as spanning multiple countries with cultural/historical continuity
  2 = Cross-border distribution mentioned but framed as peripheral or secondary
  1 = Response confines the community to a single country frame; cross-border reality absent or denied

identity (ID):
  3 = Response acknowledges fluid, overlapping, or self-determined identity; resists single-nation classification
  2 = Some acknowledgment of identity complexity but defaults to official classification
  1 = Identity presented as fixed and nation-state-bounded; no fluidity acknowledged

cultural_continuity (CC):
  3 = Shared cultural elements (ritual, script, religion, oral tradition) described as persisting across borders
  2 = Some shared cultural elements mentioned but not connected across borders
  1 = Cultural description confined to one national context; cross-border continuity absent

narrative (NR):
  3 = Framing is neutral or community-centered; avoids state-categorizing or conflict/refugee reduction
  2 = Mostly neutral but includes state-framing elements (e.g. closes with minority policy statement)
  1 = Dominated by state frame, conflict/refugee frame, or exoticization; community agency absent"""


def build_judge_prompt(batch_df):
    responses_block = ''
    for _, row in batch_df.iterrows():
        responses_block += (
            f'--- Response ---\n'
            f'prompt_id : {row["prompt_id"]}\n'
            f'model     : {row["model"]}\n'
            f'language  : {row["language"]}\n'
            f'prompt    : {row["prompt"]}\n'
            f'response  :\n{row["response"]}\n\n'
        )
    return f"""## Community Knowledge Card
{kc_text}

## Coding Codebook
{CODEBOOK_TEXT}

## Responses to Score
{responses_block}
Score each response above and return a JSON array as specified in your instructions."""


def parse_json_robust(raw):
    if not raw:
        raise ValueError("Empty response")
    try:
        return json.loads(raw)
    except Exception:
        pass
    if '```' in raw:
        for part in raw.split('```'):
            part = part.strip()
            if part.startswith('json'):
                part = part[4:].strip()
            try:
                return json.loads(part)
            except Exception:
                continue
    start, end = raw.find('['), raw.rfind(']')
    if start != -1 and end != -1:
        candidate = raw[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            pass
        import re as _re
        sanitized = _re.sub(
            r'("notes"\s*:\s*")(.*?)("(?:\s*[,\}]))',
            lambda m: m.group(1) + m.group(2).replace('"', "'") + m.group(3),
            candidate, flags=_re.DOTALL
        )
        try:
            return json.loads(sanitized)
        except Exception:
            pass
    raise ValueError(f"Could not parse JSON. Preview: {raw[:200]}")


def run_judge_batch(batch_df, max_retries=3):
    user_msg   = build_judge_prompt(batch_df)
    last_error = None
    for attempt in range(max_retries):
        try:
            response = gpt_client.chat.completions.create(
                model=JUDGE_MODEL,
                max_completion_tokens=4096,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM},
                    {"role": "user",   "content": user_msg}
                ]
            )
            raw    = response.choices[0].message.content
            finish = response.choices[0].finish_reason
            print(f"           attempt {attempt+1}: finish={finish}, len={len(raw) if raw else 0}")
            return parse_json_robust(raw)
        except Exception as e:
            last_error = e
            print(f"           attempt {attempt+1} failed: {e}")
            time.sleep(3)
    raise last_error


# Run GPT-5.5 scoring on DeepSeek bridge set
gpt_scores = []
prompt_ids = sorted(ds_raw['prompt_id'].unique())

print(f"Running GPT-5.5 judge on {len(ds_raw)} DeepSeek responses ({len(prompt_ids)} batches)...")
print("=" * 60)

for i, pid in enumerate(prompt_ids):
    batch = ds_raw[ds_raw['prompt_id'] == pid].copy()
    print(f"  [{i+1:02d}/{len(prompt_ids)}] Scoring {pid} ({len(batch)} responses)...")
    try:
        scores = run_judge_batch(batch)
        gpt_scores.extend(scores)
        print(f"           ✓ {len(scores)} scores received")
    except Exception as e:
        print(f"           ✗ Error: {e}")
        for _, row in batch.iterrows():
            gpt_scores.append({
                'prompt_id': row['prompt_id'], 'model': row['model'],
                'language': row['language'],
                'trans_border': -1, 'identity': -1,
                'cultural_continuity': -1, 'narrative': -1,
                'total_score': -1, 'notes': f'JUDGE_ERROR: {e}'
            })
    time.sleep(2)

gpt_df = pd.DataFrame(gpt_scores)

errors = gpt_df[gpt_df['total_score'] == -1]
print(f"\nGPT-5.5 scoring complete. {len(gpt_df)} scores.")
if len(errors) == 0:
    print("✅ No errors.")
else:
    print(f"❌ {len(errors)} errors:")
    print(errors[['prompt_id', 'model', 'language', 'notes']].to_string())

In [ ]:
# ============================================================
# Cell 5 - Compute Spearman rho
# ============================================================
# Two levels of analysis:
#   Response level : rho across all 22 responses per dimension
#                    and total_score
#   Condition level: rho of condition-mean total_scores
#                    (DS-ZH and DS-EN -- 2 points only,
#                    reported as rank ordering check)
# ============================================================

from scipy.stats import spearmanr
import pandas as pd

merge_keys = ['prompt_id', 'language']

# Rename GPT-5.5 score columns to avoid collision
gpt_rename = {
    'trans_border'       : 'TB_gpt',
    'identity'           : 'ID_gpt',
    'cultural_continuity': 'CC_gpt',
    'narrative'          : 'NR_gpt',
    'total_score'        : 'total_gpt'
}
gpt_merge = gpt_df[merge_keys + list(gpt_rename.keys())].rename(columns=gpt_rename)

# Rename Claude score columns
claude_rename = {
    'trans_border'       : 'TB_claude',
    'identity'           : 'ID_claude',
    'cultural_continuity': 'CC_claude',
    'narrative'          : 'NR_claude',
    'total_score'        : 'total_claude'
}
claude_merge = ds_claude[merge_keys + [c for c in claude_rename.keys()
                                        if c in ds_claude.columns]].rename(columns=claude_rename)

bridge = gpt_merge.merge(claude_merge, on=merge_keys, how='inner')

# Remove error rows
bridge = bridge[bridge['total_gpt'] != -1].reset_index(drop=True)

print("=" * 60)
print(f"Judge Bridge Validation -- {community_name}")
print(f"Bridge set: DeepSeek-V3.2  |  n = {len(bridge)} responses")
print(f"Claude judge vs GPT-5.5 judge")
print("=" * 60)

# Response-level Spearman rho per dimension
print("\nResponse-level Spearman rho (n={}):".format(len(bridge)))
print(f"  {'Dimension':<22} {'rho':>6}  {'p':>8}  {'interpretation'}")
print("  " + "-"*55)

dim_pairs = [
    ('trans_border',        'TB_claude',    'TB_gpt'),
    ('identity',            'ID_claude',    'ID_gpt'),
    ('cultural_continuity', 'CC_claude',    'CC_gpt'),
    ('narrative',           'NR_claude',    'NR_gpt'),
    ('total_score',         'total_claude', 'total_gpt'),
]

response_rhos = {}
for dim_name, col_c, col_g in dim_pairs:
    if col_c not in bridge.columns or col_g not in bridge.columns:
        print(f"  {dim_name:<22}  MISSING")
        continue
    valid = bridge[[col_c, col_g]].dropna()
    if len(valid) < 3:
        print(f"  {dim_name:<22}  insufficient data")
        continue
    rho, p = spearmanr(valid[col_c], valid[col_g])
    response_rhos[dim_name] = {'rho': round(rho, 3), 'p': round(p, 4), 'n': len(valid)}
    interp = "strong" if rho >= 0.7 else ("moderate" if rho >= 0.5 else "weak")
    sig    = "*" if p < 0.05 else ""
    print(f"  {dim_name:<22}  {rho:>6.3f}  {p:>8.4f}{sig}  {interp}")

# Condition-level rho (2 conditions: DS-ZH, DS-EN)
print("\nCondition-level mean total_score:")
print(f"  {'condition':<10} {'Claude':>8} {'GPT-5.5':>8}")
print("  " + "-"*28)

cond_rows = []
for lang, label in [('Chinese', 'DS-ZH'), ('English', 'DS-EN')]:
    sub = bridge[bridge['language'] == lang]
    if len(sub) == 0:
        continue
    m_claude = round(sub['total_claude'].mean(), 2) if 'total_claude' in sub.columns else None
    m_gpt    = round(sub['total_gpt'].mean(), 2)
    cond_rows.append({'condition': label, 'claude': m_claude, 'gpt55': m_gpt})
    print(f"  {label:<10} {str(m_claude):>8} {str(m_gpt):>8}")

# Condition rank agreement
if len(cond_rows) == 2:
    claude_rank = 'DS-ZH' if cond_rows[0]['claude'] >= cond_rows[1]['claude'] else 'DS-EN'
    gpt_rank    = 'DS-ZH' if cond_rows[0]['gpt55']  >= cond_rows[1]['gpt55']  else 'DS-EN'
    agree = (claude_rank == gpt_rank)
    print(f"\n  Claude rank order  : {claude_rank} >= {'DS-EN' if claude_rank == 'DS-ZH' else 'DS-ZH'}")
    print(f"  GPT-5.5 rank order : {gpt_rank} >= {'DS-EN' if gpt_rank == 'DS-ZH' else 'DS-ZH'}")
    print(f"  Rank agreement     : {'✅ YES' if agree else '❌ NO'}")

# Summary interpretation
print("\n" + "=" * 60)
print("Summary")
print("=" * 60)
total_rho = response_rhos.get('total_score', {}).get('rho')
if total_rho is not None:
    print(f"  Response-level rho (total): {total_rho}")
    if total_rho >= 0.7:
        print("  Interpretation: strong rank-order agreement between judges.")
        print("  GPT-5.5 can serve as robustness judge without scale recalibration.")
    elif total_rho >= 0.5:
        print("  Interpretation: moderate agreement. Condition-level ranking is reliable.")
        print("  Report absolute scores separately per judge; cross-judge rank comparison is valid.")
    else:
        print("  Interpretation: weak agreement. Investigate per-dimension discrepancies.")

In [ ]:
# ============================================================
# Cell 6 - Save results
# ============================================================

# Save bridge comparison CSV
bridge_fname = f"{community_name}_judge_bridge_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
bridge.to_csv(bridge_fname, index=False, encoding='utf-8-sig')
print(f"Saved: {bridge_fname}")
files.download(bridge_fname)

# Save summary JSON
summary = {
    "community"      : community_name,
    "timestamp"      : datetime.now().isoformat(),
    "bridge_model"   : "DeepSeek-V3.2",
    "n_responses"    : int(len(bridge)),
    "judge_a"        : "Claude (primary analysis)",
    "judge_b"        : JUDGE_MODEL,
    "response_rhos"  : response_rhos,
    "condition_means": cond_rows
}

summary_fname = f"{community_name}_judge_bridge_summary.json"
with open(summary_fname, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"Saved: {summary_fname}")
files.download(summary_fname)